# Notebook 2: Retrieval tests
**IDS 560, Group 4, M5 inventory agent (v4).** Tests how well the agent finds evidence in the library built by notebook 1. No language-model calls; runs in a few minutes.

1. **Setup:** loads the library (`chunks_v4.pkl`, `index_v4.pkl`), builds the keyword index and loads the reranker.
2. **Try a search:** `retrieve(question, date, state, category)` returns the best passages published on or before the date.
3. **The 56 test questions and the method comparison:** each retrieval method on the same tests, and the relevance cut-off for "no relevant evidence".
4. **Look at one test.**
5. **Save:** `retrieval_test_results_v4.csv`, `retrieval_methods_v4.csv`, and the cut-off in `settings_v4.json` (notebook 3 uses it).

**How `retrieve()` works** (code in `rag_common.py`):
1. **Metadata filters:** published on or before the date, the store's state or US-wide, the item's category or all categories, document age limits, only the latest version of each evidence card.
2. **Meaning + keyword search (hybrid):** ranks passages by the embedding and by the keyword index and combines the two rankings.
3. **Reranker:** reads the question with each of the top 30 passages and gives a **relevance score from 0 to 1**.
4. **Variety:** at most one passage per document and three per document type.

## 1. Setup

In [1]:
!pip -q install sentence-transformers openai pypdf
import sys, importlib
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
RAG_FILES = "/content/drive/MyDrive/IDS 560- Agentic AI & Inventory Project/RAG FILES/"   # rag_common.py lives here
sys.path.insert(0, RAG_FILES)
import rag_common as rc
importlib.reload(rc)
rc.setup(m5=False, forecasts=False, library=True)
from rag_common import *

Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (15 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Library: 7,537 passages from 5,000 documents | reranker on | RELEVANCE_MIN 0.015
Setup done in 54s | files are read from and saved to /content/drive/MyDrive/IDS 560- Agentic AI & Inventory Project/RAG FILES/ (names end in _v4)


## 2. Try a search
Change the question, date, state or category. The first number is the search score, `rel` the reranker's relevance (0 to 1).

In [2]:
show(retrieve("Why might Hobbies sales drop during a winter storm in Wisconsin?", "2015-01-06", "WI", category="HOBBIES"))

[0.033 rel 0.13] evidence_card WI 2015-01-01  Seasonal pattern: Hobbies sales in Wisconsin by weekday and month (data to 2014-12-31)
        Seasonal (weekly and yearly) sales pattern. Hobbies sales in Wisconsin M5 stores, 2011-01-29 to 2014-12-31. Index where 1.00 is an average day. Weekday: Mon 0.88, Tue 0.86, Wed 0.91, Thu 0.89, Fri 1.1...

[0.031 rel 0.12] evidence_card WI 2015-01-01  Unexpected demand spikes: Hobbies in Wisconsin (12 months to 2014-12-31)
        Demand spikes for Hobbies items in Wisconsin M5 stores, 12 months to 2014-12-31: on 5.36% of item-days (items averaging at least 0.5 units a day) sales were 3 or more times the item's previous 28-day ...

[0.025 rel 0.05] evidence_card US 2015-01-01  Event effect: Christmas (data to 2014-12-31)
        Calendar event effect: Christmas (Christmas), a national holiday. M5 sales 2011-01-29 to 2014-12-31, 4 occurrence(s) (2011, 2012, 2013, 2014). Total units on the day: -99.9% versus the weekday-adjuste...

[0.029 rel 0.03] s

## 3. Test questions and method comparison
56 questions (`TESTS` in `rag_common.py`), each with a decision date, a state and what should come back:

| Kind | Passes when |
|---|---|
| `hit` | A passage of the expected type containing one of the expected terms is in the top 5 |
| `fresh` | A Beige Book report from the last 60 days is in the top 5 |
| `no_future` | Nothing published after the date comes back |
| `no_evidence` | The question has no answer in the library: passes if the top passage is the "what the data does not contain" or "store locations" card, or every passage is below `RELEVANCE_MIN` |
| `data` | Answered from the forecast cube, not documents (tests 37–48): not scored here; notebook 3 checks the answers |

**Methods compared** (`method` in `retrieve()`): `dense` = meaning only (the v1–v2 search), `dense+category` = plus the category filter, `hybrid+category` = plus keyword search, `full` = plus the reranker.

**Columns:** share of `hit`/`fresh` tests passed; **MRR** (mean of 1 / rank of the first correct passage; 1.0 = always first); no-answer questions caught; passages from the future (must be 0).

**Relevance cut-off:** set halfway between the no-answer questions' lowest top relevance and the lower end (10th percentile) of the passed tests' top relevance, when the two separate.

In [3]:
METHODS = ["dense", "dense+category", "hybrid+category", "full"]

def run_tests(method="full", k=5):
    rows = []
    for tid, q, as_of, st, types, terms, kind, lookup in TESTS:
        if kind == "data":
            rows.append({"test": tid, "kind": kind, "method": method, "passed": None, "rank": None, "future_chunks": 0,
                         "top_relevance": None, "top_result": "(forecast-data question: checked in notebook 3)"}); continue
        res = retrieve(q, as_of, st, k=k, category=category_of(q), method=method)
        future = int((res["published"] > pd.Timestamp(as_of)).sum())
        match = res[res["doc_type"].isin(types)] if types else res
        if terms:
            pat = "|".join(terms)
            match = match[match["text"].str.lower().str.contains(pat) | match["title"].str.lower().str.contains(pat)]
        rank = (list(res.index).index(match.index[0]) + 1) if len(match) else None
        top_rel = float(res["relevance"].max()) if len(res) and res["relevance"].notna().any() else np.nan
        if kind == "hit":
            ok = rank is not None
        elif kind == "fresh":
            bb = res[res["doc_type"] == "beige_book"]
            ok = len(bb) > 0 and (pd.Timestamp(as_of) - bb["published"].max()).days <= 60
            rank = (list(res.index).index(bb.index[0]) + 1) if ok else None
        elif kind == "no_future":
            ok = future == 0
        else:                                                           # no_evidence
            ok = bool(len(res) == 0 or res.iloc[0]["series_key"] in NO_DATA_CARDS or (np.isfinite(top_rel) and top_rel < rc.RELEVANCE_MIN))
        rows.append({"test": tid, "kind": kind, "method": method, "passed": ok, "rank": rank, "future_chunks": future,
                     "top_relevance": round(top_rel, 3) if np.isfinite(top_rel) else None,
                     "top_result": res["title"].iloc[0] if len(res) else "(nothing)"})
    return pd.DataFrame(rows)

# Relevance cut-off from the tests
if rc.RERANKER is not None:
    first = run_tests("full")
    hit_rel = first.loc[(first["kind"] == "hit") & (first["passed"] == True), "top_relevance"].dropna().astype(float)
    none_rel = first.loc[first["kind"] == "no_evidence", "top_relevance"].dropna().astype(float)
    print("Top relevance: passed hit tests", hit_rel.describe()[["min", "25%", "50%"]].round(3).to_dict(),
          "| no-answer questions", sorted(none_rel.round(3).tolist()))
    if len(hit_rel) and len(none_rel) and none_rel.min() < hit_rel.quantile(0.1):
        rc.RELEVANCE_MIN = float((none_rel.min() + hit_rel.quantile(0.1)) / 2)
    print(f"RELEVANCE_MIN = {rc.RELEVANCE_MIN:.3f} (passages below it count as no relevant evidence)")

results_by_method = pd.concat([run_tests(m) for m in METHODS], ignore_index=True)
scored = results_by_method[results_by_method["kind"].isin(["hit", "fresh"])].copy()
scored["rr"] = (1 / scored["rank"].astype(float)).fillna(0)
noans = results_by_method[results_by_method["kind"] == "no_evidence"]
method_table = pd.DataFrame({
    "hit_tests_passed_%": scored.groupby("method")["passed"].apply(lambda x: 100 * x.astype(bool).mean()).round(0),
    "MRR": scored.groupby("method")["rr"].mean().round(2),
    "no_answer_caught": noans.groupby("method")["passed"].apply(lambda x: f"{int(x.astype(bool).sum())} of {len(x)}"),
    "future_passages": results_by_method.groupby("method")["future_chunks"].sum()}).reindex(METHODS)
display(method_table)
test_results = results_by_method[results_by_method["method"] == "full"].drop(columns="method").reset_index(drop=True)
display(test_results)
done = test_results[test_results["kind"] != "data"]
print(f"Full method: passed {int(done['passed'].astype(bool).sum())} of {len(done)} scored tests; "
      f"passages from the future across all methods: {int(results_by_method['future_chunks'].sum())}")

# Leakage stress test: 200 random dates and states, nothing returned may be dated after the question date
rng = np.random.default_rng(0)
bad = 0
for _ in range(200):
    d = pd.Timestamp(WINDOW_START) + pd.Timedelta(days=int(rng.integers(30, 1950)))
    res = retrieve("sales demand weather economy", d, rng.choice(list(STATES)), k=8, method="dense+category")
    bad += int((res["published"] > d).sum())
print("Leakage stress test, retrieved passages dated after the question date:", bad)

Top relevance: passed hit tests {'min': 0.001, '25%': 0.109, '50%': 0.454} | no-answer questions [0.003, 0.008, 0.015, 0.016, 0.022, 0.054, 0.071, 0.112, 0.372, 0.493]
RELEVANCE_MIN = 0.015 (passages below it count as no relevant evidence)


/tmp/ipykernel_657/3372268900.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_by_method = pd.concat([run_tests(m) for m in METHODS], ignore_index=True)


,hit_tests_passed_%,MRR,no_answer_caught,future_passages
method,,,,
dense,76.0,0.69,1 of 10,0
dense+category,76.0,0.69,1 of 10,0
hybrid+category,76.0,0.66,2 of 10,0
full,85.0,0.83,3 of 10,0


,test,kind,passed,rank,future_chunks,top_relevance,top_result
0,1,hit,False,NaN,0,0.665,"Price, sales and revenue: Hobbies in Wisconsin..."
1,2,hit,True,1.0,0,0.028,"Excessive Heat: Texas, Jun 01, 2011"
2,3,hit,True,1.0,0,0.288,"Weather effect on sales, California (data to 2..."
3,4,hit,True,1.0,0,0.498,Event effect: Christmas (data to 2013-12-31)
4,5,hit,True,1.0,0,0.210,Event effect: Christmas (data to 2013-12-31)
5,6,hit,True,1.0,0,0.959,Seasonal pattern: Sales in United States by we...
6,7,hit,True,1.0,0,0.657,SNAP benefit issue schedule by state
7,8,hit,True,1.0,0,0.773,"Beige Book January 14, 2015: National summary"
8,9,hit,True,1.0,0,0.284,How predictable item demand is: demand types (...
9,10,hit,True,1.0,0,0.994,Zero sales and missing price weeks


Full method: passed 32 of 44 scored tests; passages from the future across all methods: 0
Leakage stress test, retrieved passages dated after the question date: 0


## 4. Look at one test
The top passages for one test question with the full method. Change `TEST_ID`.

In [4]:
TEST_ID = 28
tid, q, as_of, st, types, terms, kind, lookup = next(t for t in TESTS if t[0] == TEST_ID)
print(f"Test {tid} ({kind}), as of {as_of}, state {st or 'any'}: {q}\n")
show(retrieve(q, as_of, st, k=5, category=category_of(q)))

Test 28 (hit), as of 2015-08-24, state any: How much should I raise orders before Labor Day weekend?

[0.029 rel 0.03] evidence_card US 2015-01-01  Event effect: Labor Day (data to 2014-12-31)
        Calendar event effect: Labor Day (LaborDay), a national holiday. M5 sales 2011-01-29 to 2014-12-31, 4 occurrence(s) (2011, 2012, 2013, 2014). Total units on the day: +25.0% versus the weekday-adjusted...

[0.029 rel 0.00] evidence_card TX 2015-01-01  Unexpected demand spikes: Foods in Texas (12 months to 2014-12-31)
        Demand spikes for Foods items in Texas M5 stores, 12 months to 2014-12-31: on 5.36% of item-days (items averaging at least 0.5 units a day) sales were 3 or more times the item's previous 28-day averag...

[0.031 rel 0.00] evidence_card TX 2015-01-01  Unexpected demand spikes: Household in Texas (12 months to 2014-12-31)
        Demand spikes for Household items in Texas M5 stores, 12 months to 2014-12-31: on 4.53% of item-days (items averaging at least 0.5 units a day)

## 5. Save

In [5]:
test_results.to_csv(out_path("retrieval_test_results.csv"), index=False)
method_table.to_csv(out_path("retrieval_methods.csv"))
rc.save_setting(RELEVANCE_MIN=rc.RELEVANCE_MIN)
print("Saved", os.path.basename(out_path("retrieval_test_results.csv")), "and", os.path.basename(out_path("retrieval_methods.csv")),
      f"| RELEVANCE_MIN {rc.RELEVANCE_MIN:.3f} saved to {os.path.basename(rc.SETTINGS_FILE)}")

Saved retrieval_test_results_v4.csv and retrieval_methods_v4.csv | RELEVANCE_MIN 0.015 saved to settings_v4.json
